In [1]:
import pandas as pd
import numpy as np

#cnpq_bruto = pd.read_csv("dados/cnpq_data_postgres.txt")
#cnpq_bruto.to_csv("dados/cnpq_data_postgres.csv", index=False)


### Função para normalizar os valores 

In [2]:
#
# Essa busca eu fiz no postgres, mas ela é mais eficiente no python
#
def parse_valor(valor):
    if pd.isna(valor):
        return np.nan
    
    valor = str(valor).strip()
    if not valor:
        return np.nan
    #
    # remove símbolos, alguns anos epecíficos possuem 'R$' ou '$' antes do número
    #
    valor = valor.replace('R$', '').replace('$', '').replace(' ', '')
    
    #
    # mantém apenas dígitos, vírgulas, pontos e sinais de menos
    #
    valor = ''.join(ch for ch in valor if ch.isdigit() or ch in ',.-')
    if valor in {'', '-', '.', ',', '-.', '-,'}:
        return np.nan

    ultima_virgula = valor.rfind(',')
    ultimo_ponto = valor.rfind('.')

    # escolhe separador decimal pelo último símbolo encontrado
    
    if ultima_virgula != -1 and ultimo_ponto != -1:
        if ultima_virgula > ultimo_ponto:
            normalizado = valor.replace('.', '').replace(',', '.')
        else:
            normalizado = valor.replace(',', '')
    elif ultima_virgula != -1:
        partes = valor.split(',')
        if len(partes[-1]) in (1, 2):
            normalizado = valor.replace('.', '').replace(',', '.')
        else:
            normalizado = valor.replace(',', '')
    elif ultimo_ponto != -1:
        partes = valor.split('.')
        if len(partes[-1]) in (1, 2):
            normalizado = valor.replace(',', '')
        else:
            normalizado = valor.replace('.', '')
    else:
        normalizado = valor

    try:
        return float(normalizado)
    except:
        return np.nan

### Aplica a função e exclui colunas categoricas desnecessárias 

In [3]:

cnpq_bruto = pd.read_csv("dados/cnpq_data_postgres.csv")

cnpq_bruto['valor_pago'] = cnpq_bruto['valor_pago'].apply(parse_valor)
cols_str = [
    'data_inicio_processo',
    'data_termino_processo',
    'regiao_destino',
    'titulo_do_projeto',
    'palavra_chave',
    'uo',
    'acaoppa',
    'cpf',
    'processo'
]

for col in cols_str:
    if col in cnpq_bruto.columns:
        cnpq_bruto[col] = cnpq_bruto[col].astype(str)

cols_para_excluir = [
    'data_extracao',
    'cod_modalidade',
    'id_lattes',
    'id_2020_ignorar'
    'acaoppa',
    'programappa',
    'cpf',
    'nome_chamada',
    'beneficiario',
    'processo',
    'data_inicio_processo',
    'data_termino_processo',
    'categoria_nivel',
    'uo',
    'natureza_de_despesa',

]

cnpq_bruto = cnpq_bruto.drop(columns=cols_para_excluir, errors='ignore')
cnpq_bruto = cnpq_bruto.drop(columns=['id_2020_ignorar','acaoppa'], errors='ignore')

C:\Users\felip\AppData\Local\Temp\ipykernel_8536\3586508260.py:1: DtypeWarning: Columns (0: data_inicio_processo, 1: data_termino_processo, 2: regiao_destino, 3: titulo_do_projeto, 4: palavra_chave, 5: uo, 6: valor_pago, 7: acaoppa, 8: cpf) have mixed types. Specify dtype option on import or set low_memory=False.
  cnpq_bruto = pd.read_csv("dados/cnpq_data_postgres.csv")


### Salva o arquivo -> parquet

In [5]:
import pyarrow as pa
#cnpq_bruto.to_parquet('dados/cnpqBolsasAuxilios.parquet', engine='pyarrow', index=False)
cnpq_bruto.to_parquet('dados/cnpqBolsasAuxilios.parquet', compression="zstd",  engine='pyarrow', index=False)
